<a href="https://colab.research.google.com/github/shivainlabs/Introduction-to-Deep-Learning-and-GenAI/blob/main/Week%2006/GAN_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms # used for image preprocessing
import torchvision
import os
import matplotlib.pyplot as plt

In [2]:
sample_dir = 'samples'
if not os.path.exists(sample_dir):
  os.makedirs(sample_dir)

In [3]:
latent_size = 64
hidden_size = 256
image_size = 784
num_epochs = 100

batch_size = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
device

device(type='cuda')

In [5]:
def load_mnist_data(batch_size):
  transform = transforms.Compose([
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.5],std=[0.5])
  ])

  mnist = torchvision.datasets.FashionMNIST(
      root = "./data/",
      train = True,
      transform = transform,
      download = True
  )

  data_loader = torch.utils.data.DataLoader(
      dataset = mnist,
      batch_size = batch_size,
      shuffle=True
  )

  return data_loader


In [6]:
data_loader = load_mnist_data(batch_size)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 203kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.79MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 4.38MB/s]


In [7]:
class Generator(nn.Module):
  def __init__(self,latent_size,hidden_size,image_size):
    super().__init__()

    self.model = nn.Sequential(
        nn.Linear(latent_size,hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size,hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size,image_size),
        nn.Tanh()
    )

  def forward(self,z):
    return self.model(z)

generator = Generator(latent_size,hidden_size,image_size).to(device)
generator


Generator(
  (model): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.2)
    (4): Linear(in_features=256, out_features=784, bias=True)
    (5): Tanh()
  )
)

> ```latent_size = dimension of z```





In [8]:
class Discriminator(nn.Module):
  def __init__(self,image_size,hidden_size):
    super().__init__()

    self.model = nn.Sequential(
        nn.Linear(image_size,hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size,hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size,1),
        nn.Sigmoid()
    )

  def forward(self,x):
    return self.model(x)

discriminator = Discriminator(image_size,hidden_size).to(device)
discriminator

Discriminator(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.2)
    (4): Linear(in_features=256, out_features=1, bias=True)
    (5): Sigmoid()
  )
)

In [9]:
criterion = nn.BCELoss()

d_optimizer_fixed = torch.optim.Adam(discriminator.parameters(),lr=0.0002,betas=(0.5,0.999))
g_optimizer_fixed = torch.optim.Adam(generator.parameters(),lr=0.0002,betas=(0.5,0.999))


In [10]:
def denorm(x):
  out = (x+1)/2
  return out.clamp(0,1)

def reset_grad():
  d_optimizer_fixed.zero_grad()
  g_optimizer_fixed.zero_grad()

In [11]:

print(len(data_loader.dataset)) # number of images
print(batch_size) # Images per batch


# number of batches
print(len(data_loader))
print(len(data_loader.dataset)/batch_size)

60000
100
600
600.0


In [ ]:
total_step = len(data_loader)
d_losses = []
g_losses = []
num_epochs = 100

for epoch in range(num_epochs):
  epoch_d_loss = 0
  epoch_g_loss = 0

  for i, (images,label) in enumerate(data_loader):
    images = images.reshape(batch_size,-1).to(device)

    real_labels = torch.ones(batch_size,1).to(device)
    fake_labels = torch.zeros(batch_size,1).to(device)

    outputs = discriminator(images)
    d_loss_real = criterion(outputs,real_labels)
    real_score = outputs

    z = torch.randn(batch_size,latent_size).to(device)
    fake_images = generator(z)

    outputs = discriminator(fake_images)
    d_loss_fake = criterion(outputs,fake_labels)
    fake_score = outputs

    d_loss = d_loss_real + d_loss_fake

    reset_grad()
    d_loss.backward()
    d_optimizer_fixed.step()

    # -------------------------Generator----------------------

    z = torch.rand(batch_size,latent_size).to(device)
    fake_images = generator(z)

    outputs = discriminator(fake_images)
    g_loss = criterion(outputs,real_labels)

    reset_grad()
    g_loss.backward()
    g_optimizer_fixed.step()

    epoch_d_loss += d_loss.item()
    epoch_g_loss += g_loss.item()

  d_losses.append(epoch_d_loss/total_step)
  g_losses.append(epoch_g_loss/total_step)

  fake_images_grid = fake_images.reshape(fake_images.size(0),1,28,28)
  torchvision.utils.save_image(
      denorm(fake_images_grid),
      os.path.join(sample_dir,f'fake_images-{epoch+1}.png')
  )


